# 📚 Проект 2: Классификация писателей — WriterExtnd

> **Kaggle-конкурс:** [WriterExtnd](https://www.kaggle.com/competitions/writerextnd)  
> **Задача:** Определить автора текстового фрагмента среди 6 классиков литературы  
> **Метрика оценки:** Categorical Accuracy  
> **Архитектура:** Hybrid Conv1D + Bidirectional LSTM

| Label | Автор |
|-------|-------|
| 0 | О. Генри |
| 1 | К. Саймак |
| 2 | М. Булгаков |
| 3 | Р. Брэдбери |
| 4 | М. Фрай |
| 5 | Братья Стругацкие |

## 📋 Содержание
1. [Установка зависимостей и загрузка датасета](#setup)
2. [Предобработка данных](#preprocessing)
3. [Разведывательный анализ (EDA)](#eda)
4. [Подготовка обучающей выборки](#dataset)
5. [Построение и обучение модели](#model)
6. [Оценка качества](#evaluation)
7. [XAI: Интерпретация предсказаний](#xai)
8. [Генерация submission.csv](#submission)

---
## 1. Установка зависимостей и загрузка датасета <a name='setup'></a>

### Автоматическая загрузка с Kaggle
Для работы необходим файл `kaggle.json` с API-токеном. Получить его можно на странице профиля Kaggle:
`Account → API → Create New Token`

In [ ]:
# ─── Установка необходимых библиотек ───────────────────────────────────────────
!pip install -q kaggle lime

In [ ]:
import os
import json
from google.colab import files

# ─── Настройка Kaggle API ───────────────────────────────────────────────────────
# Вариант 1: загрузить kaggle.json вручную
print('Загрузите файл kaggle.json (Account → API → Create New Token)')
uploaded = files.upload()

# Перемещаем токен в правильное место
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(list(uploaded.values())[0].decode())
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ kaggle.json настроен')

In [ ]:
# ─── Загрузка датасета WriterExtnd с Kaggle ─────────────────────────────────────
# Скачиваем файлы конкурса одной командой
!kaggle competitions download -c writerextnd -p /content/data/

# Распаковываем архив
!cd /content/data && unzip -o '*.zip' && ls -la

DATA_DIR = '/content/data'
print(f'\n📁 Содержимое папки с данными:')
!ls -la /content/data/

In [ ]:
# ─── Импорт всех необходимых библиотек ─────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import re
import os
import warnings
from collections import Counter

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, MaxPooling1D,
    Bidirectional, LSTM, Dense, Dropout,
    GlobalMaxPooling1D, BatchNormalization, Add
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'

# Проверяем доступность GPU
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU доступен: {len(gpus) > 0} → {gpus}')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Словарь меток авторов
AUTHOR_MAP = {
    0: 'О. Генри',
    1: 'К. Саймак',
    2: 'М. Булгаков',
    3: 'Р. Брэдбери',
    4: 'М. Фрай',
    5: 'Стругацкие'
}
COLORS = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6', '#1ABC9C']

---
## 2. Предобработка данных <a name='preprocessing'></a>

### Стратегия обработки
1. **Нормализация текста** — приведение к нижнему регистру, удаление пунктуации и цифр
2. **Пословная нарезка (Word Sliding Window)** — формирование фрагментов по 300 слов с шагом 150 (50% перекрытие)
3. **Баланс классов** — применение `class_weight` для компенсации дисбаланса

In [ ]:
def clean_text(text: str) -> str:
    """Нормализация текста:
    - Приведение к нижнему регистру
    - Удаление пунктуации, цифр и спецсимволов
    - Убираем лишние пробелы
    Сохраняем русские и латинские буквы, чтобы поддержать оба языка.
    """
    text = text.lower()
    text = re.sub(r'[^а-яёa-z\s]', '', text)  # только буквы
    text = re.sub(r'\s+', ' ', text).strip()  # нормализация пробелов
    return text


def prepare_data(folder: str, chunk_size: int = 300, step: int = 150) -> pd.DataFrame:
    """Загружает тексты авторов и нарезает их скользящим окном по словам.

    Args:
        folder: путь к папке с txt-файлами авторов
        chunk_size: размер окна в словах (300 слов ≈ 1 страница книги)
        step: шаг окна в словах (150 = 50% перекрытие → аугментация данных)

    Returns:
        DataFrame с колонками ['text', 'label']
    """
    # Маппинг файлов к числовым меткам классов
    authors_map = {
        'Genri.txt':       0,
        'Simak.txt':       1,
        'Bulgakov.txt':    2,
        'Bradbury.txt':    3,
        'Fry.txt':         4,
        'Strugatskie.txt': 5,
    }

    records = []
    print('─' * 55)
    print(f'{'Автор':<20} {'Слов':>8} {'Фрагментов':>12}')
    print('─' * 55)

    for filename, label in authors_map.items():
        path = os.path.join(folder, filename)
        if not os.path.exists(path):
            print(f'  ⚠️  Файл не найден: {path}')
            continue

        with open(path, 'r', encoding='utf-8') as f:
            raw = f.read()

        # 1. Очистка всего текста
        cleaned = clean_text(raw)
        # 2. Разбиваем на список слов
        words = cleaned.split()
        # 3. Нарезка скользящим окном
        count = 0
        for i in range(0, len(words) - chunk_size + 1, step):
            chunk = ' '.join(words[i: i + chunk_size])
            records.append({'text': chunk, 'label': label})
            count += 1

        print(f'  {AUTHOR_MAP[label]:<18} {len(words):>8,} {count:>12,}')

    print('─' * 55)
    df = pd.DataFrame(records)
    print(f'  ИТОГО фрагментов: {len(df):,}')
    return df


# ─── Загрузка и подготовка данных ───────────────────────────────────────────────
train_df = prepare_data(DATA_DIR, chunk_size=300, step=150)
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'\nРазмер датасета: {train_df.shape}')
train_df.head(3)

---
## 3. Разведывательный анализ данных (EDA) <a name='eda'></a>

In [ ]:
# ─── 3.1 Анализ баланса классов ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Распределение данных по авторам', fontsize=14, fontweight='bold', y=1.02)

counts = train_df['label'].value_counts().sort_index()
author_names = [AUTHOR_MAP[i] for i in counts.index]

# Bar chart
bars = axes[0].bar(author_names, counts.values, color=COLORS, edgecolor='white', linewidth=1.5)
axes[0].set_title('Количество фрагментов на автора', fontweight='bold')
axes[0].set_ylabel('Количество фрагментов')
axes[0].tick_params(axis='x', rotation=25)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=author_names, colors=COLORS,
    autopct='%1.1f%%', startangle=90, pctdistance=0.8,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(9)
axes[1].set_title('Доля каждого автора (%)', fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Статистика дисбаланса
print(f'\n📊 Статистика баланса классов:')
print(f'   Максимум: {counts.max():,} фрагментов → {AUTHOR_MAP[counts.idxmax()]}')
print(f'   Минимум:  {counts.min():,} фрагментов → {AUTHOR_MAP[counts.idxmin()]}')
print(f'   Коэффициент дисбаланса: {counts.max()/counts.min():.2f}x')
print('\n   ⚠️  Для компенсации дисбаланса будут применены class_weights')

In [ ]:
# ─── 3.2 Анализ длин фрагментов (в токенах/словах) ─────────────────────────────
train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))
train_df['char_count'] = train_df['text'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Распределение длин текстовых фрагментов', fontsize=14, fontweight='bold')

# Гистограмма длин в словах
axes[0].hist(train_df['word_count'], bins=40, color='#3498DB', edgecolor='white', alpha=0.8)
axes[0].axvline(train_df['word_count'].quantile(0.95), color='red',
                linestyle='--', linewidth=2, label=f'P95 = {train_df["word_count"].quantile(0.95):.0f}')
axes[0].axvline(train_df['word_count'].median(), color='orange',
                linestyle='--', linewidth=2, label=f'Median = {train_df["word_count"].median():.0f}')
axes[0].set_title('Длина фрагментов (в словах)')
axes[0].set_xlabel('Количество слов')
axes[0].legend()

# Боксплот по авторам
data_by_author = [train_df[train_df['label'] == i]['word_count'].values for i in range(6)]
bp = axes[1].boxplot(data_by_author, patch_artist=True, labels=[AUTHOR_MAP[i] for i in range(6)])
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Длина фрагментов по авторам')
axes[1].set_ylabel('Количество слов')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig('text_length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Статистика длин фрагментов (слова):')
print(train_df['word_count'].describe().round(1))

In [ ]:
# ─── 3.3 Анализ словарного состава — топ-20 слов каждого автора ─────────────────
from collections import Counter

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Топ-20 слов каждого автора (без стоп-слов)', fontsize=14, fontweight='bold')

# Базовые стоп-слова для русского языка
STOP_WORDS = set([
    'и', 'в', 'не', 'он', 'на', 'я', 'что', 'тот', 'быть', 'с', 'а',
    'весь', 'это', 'как', 'она', 'по', 'но', 'они', 'к', 'у', 'ты',
    'из', 'мы', 'за', 'то', 'так', 'вот', 'от', 'до', 'же', 'или',
    'при', 'был', 'была', 'было', 'были', 'чтобы', 'ещё', 'уже', 'для',
    'если', 'его', 'её', 'их', 'мне', 'him', 'the', 'a', 'of', 'to',
    'and', 'in', 'that', 'he', 'it', 'with', 'as', 'his', 'for', 'was'
])

for idx, (label, ax) in enumerate(zip(range(6), axes.flatten())):
    subset = train_df[train_df['label'] == label]['text']
    all_words = ' '.join(subset.values).split()
    # Фильтруем стоп-слова и короткие слова
    filtered = [w for w in all_words if w not in STOP_WORDS and len(w) > 3]
    top_words = Counter(filtered).most_common(20)

    words_list, counts_list = zip(*top_words)
    bars = ax.barh(range(len(words_list)), counts_list, color=COLORS[idx], alpha=0.85)
    ax.set_yticks(range(len(words_list)))
    ax.set_yticklabels(words_list, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(f'{AUTHOR_MAP[label]}', fontweight='bold', color=COLORS[idx])
    ax.set_xlabel('Частота')

plt.tight_layout()
plt.savefig('top_words_per_author.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Подготовка обучающей выборки <a name='dataset'></a>

**Параметры токенизации:**
- `VOCAB_SIZE = 50000` — размер словаря (топ-50k наиболее частых слов)
- `MAX_LEN = 300` — максимальная длина последовательности (P95 распределения)
- `EMBED_DIM = 128` — размерность эмбеддинга

In [ ]:
# ─── Гиперпараметры токенизации ─────────────────────────────────────────────────
VOCAB_SIZE = 50_000   # топ-N слов словаря
MAX_LEN    = 300      # длина последовательности (≈ P95)
EMBED_DIM  = 128      # размерность векторного представления слов
NUM_CLASSES = 6

# ─── Разбивка на train/validation ───────────────────────────────────────────────
X_raw = train_df['text'].values
y_raw = train_df['label'].values

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw, y_raw,
    test_size=0.15,
    random_state=SEED,
    stratify=y_raw  # стратификация — сохраняем распределение классов
)

print(f'Train: {len(X_train_raw):,} | Val: {len(X_val_raw):,}')

# ─── Токенизация ────────────────────────────────────────────────────────────────
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token='<OOV>',  # токен для слов вне словаря
    filters=''          # уже очистили текст в clean_text
)
tokenizer.fit_on_texts(X_train_raw)  # строим словарь только по train!

actual_vocab = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
print(f'Реальный размер словаря: {actual_vocab:,}')

# ─── Паддинг последовательностей ────────────────────────────────────────────────
X_train = pad_sequences(
    tokenizer.texts_to_sequences(X_train_raw),
    maxlen=MAX_LEN, padding='post', truncating='post'
)
X_val = pad_sequences(
    tokenizer.texts_to_sequences(X_val_raw),
    maxlen=MAX_LEN, padding='post', truncating='post'
)

print(f'X_train shape: {X_train.shape}')
print(f'X_val shape:   {X_val.shape}')

# ─── Веса классов для компенсации дисбаланса ────────────────────────────────────
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights_arr))

print('\nВеса классов (class_weight):')
for label, weight in class_weights.items():
    print(f'  {AUTHOR_MAP[label]:<18}: {weight:.3f}')

---
## 5. Построение и обучение модели <a name='model'></a>

### Архитектура: Hybrid Conv1D + Bidirectional LSTM

```
Input (MAX_LEN,)
    │
Embedding (VOCAB_SIZE × EMBED_DIM)
    │
    ├─── Conv1D(128, kernel=3) → BatchNorm → MaxPool
    │         │
    │    Conv1D(128, kernel=5) → BatchNorm → MaxPool
    │         │
    └─── Add → Concat → BiLSTM(128) → Dropout(0.4)
                                │
                         Dense(128, relu) → Dropout(0.3)
                                │
                         Dense(6, softmax)
```

**Обоснование:**
- **Embedding** — обучаемые векторные представления слов
- **Conv1D с разными kernel** — параллельное извлечение n-граммных признаков (3 и 5 слов)
- **Bidirectional LSTM** — улавливает зависимости слева и справа для понимания синтаксиса
- **BatchNormalization** — стабилизация обучения
- **Dropout** — регуляризация для предотвращения переобучения

In [ ]:
def build_model(vocab_size: int, embed_dim: int, max_len: int, num_classes: int) -> Model:
    """Гибридная архитектура Conv1D + Bidirectional LSTM для классификации текстов.

    Параллельные Conv1D с разными kernel_size позволяют одновременно
    извлекать n-граммные признаки разного масштаба.
    LSTM поверх конволюционных признаков улавливает дальние зависимости.
    """
    inp = Input(shape=(max_len,), name='input_ids')

    # Обучаемый слой эмбеддингов
    x = Embedding(vocab_size, embed_dim, name='embedding')(inp)
    # Небольшой dropout после эмбеддинга — регуляризация
    x = Dropout(0.2)(x)

    # ── Ветка 1: Conv1D kernel=3 (триграммы) ──────────────────────────────────
    c3 = Conv1D(128, kernel_size=3, activation='relu', padding='same', name='conv_k3')(x)
    c3 = BatchNormalization()(c3)
    c3 = MaxPooling1D(pool_size=2, name='pool_k3')(c3)

    # ── Ветка 2: Conv1D kernel=5 (пятиграммы) ─────────────────────────────────
    c5 = Conv1D(128, kernel_size=5, activation='relu', padding='same', name='conv_k5')(x)
    c5 = BatchNormalization()(c5)
    c5 = MaxPooling1D(pool_size=2, name='pool_k5')(c5)

    # ── Объединение признаков из двух веток (сложение) ────────────────────────
    # Add работает лучше Concatenate при одинаковой размерности, т.к.
    # позволяет LSTM учиться на сумме признаков без удвоения размерности
    merged = Add(name='merge_features')([c3, c5])

    # ── Bidirectional LSTM — захват долгосрочных зависимостей ─────────────────
    x = Bidirectional(LSTM(128, return_sequences=False), name='bilstm')(merged)
    x = Dropout(0.4, name='drop_lstm')(x)

    # ── Полносвязные слои классификации ───────────────────────────────────────
    x = Dense(128, activation='relu', name='dense_1')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3, name='drop_dense')(x)

    # Выходной слой: softmax для многоклассовой классификации
    out = Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inp, outputs=out, name='WriterClassifier')
    return model


# ─── Компиляция модели ──────────────────────────────────────────────────────────
model = build_model(
    vocab_size=actual_vocab,
    embed_dim=EMBED_DIM,
    max_len=MAX_LEN,
    num_classes=NUM_CLASSES
)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',  # метки в виде целых чисел (не one-hot)
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ─── Гиперпараметры обучения ────────────────────────────────────────────────────
BATCH_SIZE = 64
EPOCHS     = 40

# ─── Callbacks ──────────────────────────────────────────────────────────────────
callbacks = [
    # Останавливаем обучение, если val_loss не улучшается 5 эпох подряд
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # Уменьшаем LR в 2 раза, если нет прогресса 3 эпохи
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    # Сохраняем лучшую модель
    ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# ─── Обучение ───────────────────────────────────────────────────────────────────
print('🚀 Начало обучения...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,  # компенсация дисбаланса
    callbacks=callbacks,
    verbose=1
)
print('✅ Обучение завершено!')

In [ ]:
# ─── Кривые обучения ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Кривые обучения модели', fontsize=14, fontweight='bold')

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train Accuracy', color='#3498DB', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   color='#E74C3C', linewidth=2)
axes[0].set_title('Точность (Accuracy)')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train Loss', color='#3498DB', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='#E74C3C', linewidth=2)
axes[1].set_title('Функция потерь (Loss)')
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

best_val_acc = max(history.history['val_accuracy'])
print(f'🏆 Лучшая Val Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')

---
## 6. Оценка качества модели <a name='evaluation'></a>

In [ ]:
# ─── Загружаем лучшую модель и получаем предсказания ────────────────────────────
from tensorflow.keras.models import load_model
model = load_model('best_model.h5')

y_pred_proba = model.predict(X_val, batch_size=128, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

# ─── Итоговые метрики ──────────────────────────────────────────────────────────
acc   = accuracy_score(y_val, y_pred)
f1_w  = f1_score(y_val, y_pred, average='weighted')
f1_m  = f1_score(y_val, y_pred, average='macro')

print('=' * 60)
print(f'  Accuracy (Val):        {acc:.4f}  ({acc*100:.2f}%)')
print(f'  F1-score (weighted):   {f1_w:.4f}')
print(f'  F1-score (macro):      {f1_m:.4f}')
print('=' * 60)

# ─── Детальный отчёт по классам ────────────────────────────────────────────────
print('\nОтчёт по классам:')
target_names = [AUTHOR_MAP[i] for i in range(NUM_CLASSES)]
print(classification_report(y_val, y_pred, target_names=target_names))

In [ ]:
# ─── Матрица ошибок ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Матрица ошибок (Confusion Matrix)', fontsize=14, fontweight='bold')

cm        = confusion_matrix(y_val, y_pred)
cm_norm   = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # нормализованная
tick_labels = [AUTHOR_MAP[i] for i in range(NUM_CLASSES)]

# Абсолютные значения
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=tick_labels, yticklabels=tick_labels, ax=axes[0])
axes[0].set_title('Абсолютные значения')
axes[0].set_ylabel('Истинный класс')
axes[0].set_xlabel('Предсказанный класс')
axes[0].tick_params(axis='x', rotation=30)

# Нормализованная (вероятности)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=tick_labels, yticklabels=tick_labels,
            vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('Нормализованная (recall per class)')
axes[1].set_ylabel('Истинный класс')
axes[1].set_xlabel('Предсказанный класс')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Анализ наиболее частых ошибок
print('\n🔍 Топ-5 пар с наибольшим числом ошибок (истинный → предсказанный):')
errors = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] > 0:
            errors.append((cm[i, j], AUTHOR_MAP[i], AUTHOR_MAP[j]))
for cnt, true, pred in sorted(errors, reverse=True)[:5]:
    print(f'  {true:<18} → {pred:<18}: {cnt:3d} ошибок')

---
## 7. XAI: Интерпретация предсказаний <a name='xai'></a>

Используем **LIME (Local Interpretable Model-Agnostic Explanations)** для объяснения конкретных предсказаний модели — это позволяет понять, какие слова "сигнализируют" на конкретного автора.

In [ ]:
from lime.lime_text import LimeTextExplainer

# ─── Обёртка predict для LIME (принимает сырые тексты) ──────────────────────────
def predict_proba_lime(texts):
    """Принимает список сырых строк → возвращает массив вероятностей."""
    sequences = pad_sequences(
        tokenizer.texts_to_sequences([clean_text(t) for t in texts]),
        maxlen=MAX_LEN, padding='post', truncating='post'
    )
    return model.predict(sequences, verbose=0)


# ─── Инициализируем LIME ────────────────────────────────────────────────────────
explainer = LimeTextExplainer(
    class_names=[AUTHOR_MAP[i] for i in range(NUM_CLASSES)]
)

# ─── Выбираем 2 примера: правильное и ошибочное предсказание ───────────────────
correct_mask = (y_pred == y_val)
wrong_mask   = ~correct_mask

idx_correct = np.where(correct_mask)[0][0]
idx_wrong   = np.where(wrong_mask)[0][0] if wrong_mask.any() else idx_correct + 1

for name, idx in [('✅ Правильное предсказание', idx_correct),
                   ('❌ Ошибочное предсказание',  idx_wrong)]:
    sample_text = X_val_raw[idx]
    true_label  = y_val[idx]
    pred_label  = y_pred[idx]

    print(f'\n{name}')
    print(f'  Истинный автор:     {AUTHOR_MAP[true_label]}')
    print(f'  Предсказанный автор:{AUTHOR_MAP[pred_label]}')

    exp = explainer.explain_instance(
        sample_text,
        predict_proba_lime,
        num_features=15,
        labels=[true_label]
    )
    exp.save_to_file(f'lime_explanation_{name[:2]}.html')

    # Визуализируем топ-15 слов-признаков
    features = exp.as_list(label=true_label)
    words = [f[0] for f in features]
    importances = [f[1] for f in features]
    colors_lime = ['#2ECC71' if v > 0 else '#E74C3C' for v in importances]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(range(len(words)), importances, color=colors_lime, alpha=0.85)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'LIME объяснение: {name}\n(Зелёный = за автора, Красный = против)',
                 fontweight='bold')
    ax.set_xlabel('Важность признака')
    plt.tight_layout()
    plt.savefig(f'lime_{name[:2]}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ─── Тепловая карта уверенности модели по тестовым файлам ──────────────────────
# Загружаем тестовые файлы (author0.txt … author20.txt)
test_files = sorted(
    [f for f in os.listdir(DATA_DIR) if f.startswith('author') and f.endswith('.txt')],
    key=lambda x: int(x.replace('author', '').replace('.txt', ''))
)

if test_files:
    test_texts = []
    for fname in test_files:
        with open(os.path.join(DATA_DIR, fname), 'r', encoding='utf-8') as f:
            test_texts.append(clean_text(f.read()))

    test_seq  = pad_sequences(
        tokenizer.texts_to_sequences(test_texts),
        maxlen=MAX_LEN, padding='post', truncating='post'
    )
    test_proba = model.predict(test_seq, verbose=0)

    # Тепловая карта вероятностей
    fig, ax = plt.subplots(figsize=(10, max(6, len(test_files) * 0.4)))
    sns.heatmap(
        test_proba,
        annot=True, fmt='.2f', cmap='YlOrRd',
        xticklabels=[AUTHOR_MAP[i] for i in range(NUM_CLASSES)],
        yticklabels=[f.replace('.txt', '') for f in test_files],
        vmin=0, vmax=1, ax=ax
    )
    ax.set_title('Уверенность модели (вероятности классов) для тестовых файлов',
                 fontweight='bold')
    ax.set_xlabel('Автор')
    ax.set_ylabel('Тестовый файл')
    plt.tight_layout()
    plt.savefig('prediction_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  Тестовые файлы (author*.txt) не найдены в папке DATA_DIR')

---
## 8. Генерация submission.csv <a name='submission'></a>

In [ ]:
# ─── Генерация submission.csv ────────────────────────────────────────────────────
# Ищем тестовые файлы по паттерну author*.txt
test_files = sorted(
    [f for f in os.listdir(DATA_DIR) if f.startswith('author') and f.endswith('.txt')],
    key=lambda x: int(x.replace('author', '').replace('.txt', ''))
)

if not test_files:
    raise FileNotFoundError('Тестовые файлы не найдены! Проверьте путь DATA_DIR')

print(f'Найдено тестовых файлов: {len(test_files)}')

submission_rows = []

for fname in test_files:
    fpath = os.path.join(DATA_DIR, fname)
    with open(fpath, 'r', encoding='utf-8') as f:
        raw_text = f.read()

    # Очищаем текст и токенизируем
    cleaned  = clean_text(raw_text)
    seq      = pad_sequences(
        tokenizer.texts_to_sequences([cleaned]),
        maxlen=MAX_LEN, padding='post', truncating='post'
    )

    # Получаем предсказание (argmax из вектора вероятностей)
    proba       = model.predict(seq, verbose=0)
    pred_label  = int(np.argmax(proba, axis=1)[0])
    confidence  = float(np.max(proba))

    doc_id = fname.replace('.txt', '')  # 'author1', 'author2', ...
    submission_rows.append({
        'id':    doc_id,
        'label': pred_label
    })
    print(f'  {doc_id:<12}: {AUTHOR_MAP[pred_label]:<18} (conf: {confidence:.3f})')

# ─── Сохраняем submission.csv ────────────────────────────────────────────────────
submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv('submission.csv', index=False)

print(f'\n✅ submission.csv сохранён!')
print(submission_df.head(10))

In [ ]:
# ─── Сохранение всех артефактов ─────────────────────────────────────────────────
import pickle

# Сохраняем токенизатор для воспроизводимости
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Сохраняем финальную модель
model.save('writer_classifier_final.h5')

print('Сохранены файлы:')
print('  📄 submission.csv        — ответы для Kaggle')
print('  🤖 best_model.h5         — лучшая модель по val_accuracy')
print('  🤖 writer_classifier_final.h5 — финальная модель')
print('  🔤 tokenizer.pkl         — токенизатор')
print('  📊 training_curves.png   — кривые обучения')
print('  📊 confusion_matrix.png  — матрица ошибок')
print('  📊 class_distribution.png')
print('  📊 top_words_per_author.png')

In [ ]:
# ─── Скачивание результатов (для Google Colab) ──────────────────────────────────
from google.colab import files

files.download('submission.csv')
files.download('training_curves.png')
files.download('confusion_matrix.png')
print('✅ Файлы загружены!')